In [6]:
!pip install -q ultralytics onnxruntime easyocr fastapi uvicorn python-multipart requests pyngrok


In [7]:
from google.colab import drive
from pathlib import Path
import zipfile
import sys
import torch

drive.mount("/content/drive")

BASE = Path(
    "/content/drive/MyDrive/8vo semestre /Machine learning /PeajeInteligente"
)

assert BASE.is_dir(), "No se encontró la carpeta del proyecto"

M1_ZIP = BASE / "modelos/placas-web_v2 (1).zip"
M2_PATH = BASE / "modelos/M2_vehiculos/weights/best.pt"
M3_PATH = BASE / "modelos/M3_llantas/weights/best.pt"

assert M1_ZIP.is_file(), "No se encontró M1"
assert M2_PATH.is_file(), "No se encontró M2"
assert M3_PATH.is_file(), "No se encontró M3"

RUTA_M1 = Path("/content/recursos_m1")

with zipfile.ZipFile(M1_ZIP, "r") as z:
    z.extractall(RUTA_M1)

DETECTOR_PATH = RUTA_M1 / "placas-web/python/placas"

sys.path.insert(0, str(DETECTOR_PATH))

assert (DETECTOR_PATH / "detector.py").is_file()

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", DEVICE)
print("M1:", M1_ZIP)
print("M2:", M2_PATH)
print("M3:", M3_PATH)

print("LOS TRES MODELOS ESTÁN DISPONIBLES")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cpu
M1: /content/drive/MyDrive/8vo semestre /Machine learning /PeajeInteligente/modelos/placas-web_v2 (1).zip
M2: /content/drive/MyDrive/8vo semestre /Machine learning /PeajeInteligente/modelos/M2_vehiculos/weights/best.pt
M3: /content/drive/MyDrive/8vo semestre /Machine learning /PeajeInteligente/modelos/M3_llantas/weights/best.pt
LOS TRES MODELOS ESTÁN DISPONIBLES


In [8]:
import cv2
import numpy as np
import easyocr
import onnxruntime as ort
import detector
import re
import time

from ultralytics import YOLO

# M1: Detector de placas

sesion_m1 = ort.InferenceSession(
    str(detector.MODELO),
    providers=["CPUExecutionProvider"]
)

lector_ocr = easyocr.Reader(
    ["en"],
    gpu=torch.cuda.is_available(),
    verbose=False
)

# M2: Segmentación de vehículos

modelo_m2 = YOLO(str(M2_PATH))
modelo_m2.to(DEVICE)

# M3: Keypoints de ruedas

modelo_m3 = YOLO(str(M3_PATH))
modelo_m3.to(DEVICE)

print("M1 cargado correctamente")
print("M2 cargado correctamente")
print("M3 cargado correctamente")

print("Clases M2:", modelo_m2.names)
print("Clases M3:", modelo_m3.names)

print("Dispositivo:", DEVICE)
print("MODELOS LISTOS PARA INFERENCIA")

M1 cargado correctamente
M2 cargado correctamente
M3 cargado correctamente
Clases M2: {0: 'auto', 1: 'bus', 2: 'camion', 3: 'minibus', 4: 'moto'}
Clases M3: {0: 'llanta'}
Dispositivo: cpu
MODELOS LISTOS PARA INFERENCIA


In [9]:
import cv2
import numpy as np
import re
import time

PERMITIDOS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

FORMATO_BOLIVIA = re.compile(r"^\d{3,4}[A-Z]{3}$")


# =====================================
# NORMALIZACIÓN DE PLACAS
# =====================================

def normalizar_placa(texto):

    return re.sub(
        r"[^A-Z0-9]",
        "",
        texto.upper()
    )


# =====================================
# M1: DETECCIÓN DE PLACAS
# =====================================

def detectar_placas(imagen, umbral=0.4):

    alto, ancho = imagen.shape[:2]

    lado = 384

    escala = min(lado / alto, lado / ancho)

    nuevo_ancho = round(ancho * escala)
    nuevo_alto = round(alto * escala)

    margen_x = (lado - nuevo_ancho) / 2
    margen_y = (lado - nuevo_alto) / 2

    lienzo = cv2.resize(
        imagen,
        (nuevo_ancho, nuevo_alto)
    )

    lienzo = cv2.copyMakeBorder(
        lienzo,
        round(margen_y - 0.1),
        round(margen_y + 0.1),
        round(margen_x - 0.1),
        round(margen_x + 0.1),
        cv2.BORDER_CONSTANT,
        value=(114, 114, 114)
    )

    entrada = (
        lienzo[:, :, ::-1].transpose(2, 0, 1)
        / 255.0
    ).astype(np.float32)[None]

    salida = sesion_m1.run(
        None,
        {
            sesion_m1.get_inputs()[0].name: entrada
        }
    )[0]

    cajas = []

    for _, x1, y1, x2, y2, _, confianza in salida:

        if confianza < umbral:
            continue

        x1 = int(max(0, (x1 - margen_x) / escala))
        y1 = int(max(0, (y1 - margen_y) / escala))

        x2 = int(min(ancho, (x2 - margen_x) / escala))
        y2 = int(min(alto, (y2 - margen_y) / escala))

        if x2 > x1 and y2 > y1:

            cajas.append(
                (x1, y1, x2, y2, float(confianza))
            )

    return cajas


# =====================================
# M1: LECTURA OCR
# =====================================

def inferencia_m1(imagen):

    cajas = detectar_placas(imagen)

    candidatos = []

    for x1, y1, x2, y2, conf_det in cajas:

        recorte = imagen[y1:y2, x1:x2]

        if recorte.size == 0:
            continue

        lecturas = lector_ocr.readtext(
            recorte,
            allowlist=PERMITIDOS
        )

        if not lecturas:
            continue

        def altura(lectura):

            return (
                max(p[1] for p in lectura[0])
                - min(p[1] for p in lectura[0])
            )

        maxima = max(altura(l) for l in lecturas)

        lecturas = [
            l for l in lecturas
            if altura(l) >= maxima * 0.6
        ]

        lecturas.sort(
            key=lambda l: l[0][0][0]
        )

        texto = "".join(
            l[1] for l in lecturas
        )

        placa = normalizar_placa(texto)

        confianza = sum(
            float(l[2]) for l in lecturas
        ) / len(lecturas)

        if placa:

            candidatos.append({
                "placa": placa,
                "confianza": round(confianza, 4),
                "confianza_deteccion": round(conf_det, 4),
                "formato_valido": bool(
                    FORMATO_BOLIVIA.fullmatch(placa)
                ),
                "bbox": [x1, y1, x2, y2]
            })

    if not candidatos:

        return {
            "placa": None,
            "confianza": 0.0,
            "confianza_deteccion": 0.0,
            "formato_valido": False,
            "bbox": None
        }

    candidatos.sort(
        key=lambda x: (
            x["formato_valido"],
            x["confianza"]
        ),
        reverse=True
    )

    return candidatos[0]


# =====================================
# M2: SEGMENTACIÓN DE VEHÍCULOS
# =====================================

def inferencia_m2(imagen):

    resultado = modelo_m2.predict(
        source=imagen,
        conf=0.35,
        imgsz=640,
        device=DEVICE,
        verbose=False
    )[0]

    if resultado.boxes is None or len(resultado.boxes) == 0:

        return {
            "categoria": None,
            "confianza_categoria": 0.0,
            "bbox": None,
            "mascara": None
        }

    cajas = resultado.boxes.xyxy.cpu().numpy()

    areas = (
        (cajas[:, 2] - cajas[:, 0])
        * (cajas[:, 3] - cajas[:, 1])
    )

    indice = int(np.argmax(areas))

    caja = resultado.boxes[indice]

    clase = int(caja.cls.item())

    categoria = str(
        modelo_m2.names[clase]
    ).lower()

    bbox = [
        int(v) for v in cajas[indice]
    ]

    mascara = None

    if resultado.masks is not None:

        mascara = (
            resultado.masks.xy[indice]
            .astype(int)
            .tolist()
        )

    return {
        "categoria": categoria,
        "confianza_categoria": round(
            float(caja.conf.item()), 4
        ),
        "bbox": bbox,
        "mascara": mascara
    }


# =====================================
# M3: DETECCIÓN DE RUEDAS Y EJES
# =====================================

def inferencia_m3(imagen, vehiculo):

    bbox = vehiculo.get("bbox")

    if bbox is None:

        return {
            "ejes": None,
            "ruedas": [],
            "confianza_ejes": 0.0
        }

    x1, y1, x2, y2 = bbox

    alto, ancho = imagen.shape[:2]

    margen = int(
        max(x2 - x1, y2 - y1) * 0.05
    )

    x1 = max(0, x1 - margen)
    y1 = max(0, y1 - margen)

    x2 = min(ancho, x2 + margen)
    y2 = min(alto, y2 + margen)

    recorte = imagen[y1:y2, x1:x2]

    if recorte.size == 0:

        return {
            "ejes": None,
            "ruedas": [],
            "confianza_ejes": 0.0
        }

    resultado = modelo_m3.predict(
        source=recorte,
        conf=0.25,
        iou=0.45,
        imgsz=640,
        device=DEVICE,
        verbose=False
    )[0]

    ruedas = []

    if (
        resultado.keypoints is not None
        and resultado.boxes is not None
    ):

        puntos = (
            resultado.keypoints.data
            .cpu()
            .numpy()
        )

        confianzas = (
            resultado.boxes.conf
            .cpu()
            .numpy()
        )

        for kp, conf in zip(puntos, confianzas):

            if len(kp) == 0:
                continue

            centro = kp[0]

            if len(centro) > 2 and centro[2] < 0.25:
                continue

            cx = float(centro[0])
            cy = float(centro[1])

            if cx <= 0 or cy <= 0:
                continue

            ruedas.append({
                "x": round(cx + x1, 2),
                "y": round(cy + y1, 2),
                "confianza": round(float(conf), 4)
            })

    ruedas.sort(
        key=lambda r: r["confianza"],
        reverse=True
    )

    # Agrupar ruedas con posiciones horizontales
    # muy cercanas para evitar duplicar ejes.

    ruedas_unicas = []

    tolerancia = max(
        8,
        (x2 - x1) * 0.035
    )

    for rueda in ruedas:

        duplicada = any(
            abs(rueda["x"] - otra["x"]) < tolerancia
            for otra in ruedas_unicas
        )

        if not duplicada:
            ruedas_unicas.append(rueda)

    ruedas_unicas.sort(
        key=lambda r: r["x"]
    )

    ejes = (
        len(ruedas_unicas)
        if len(ruedas_unicas) >= 2
        else None
    )

    confianza = (
        sum(r["confianza"] for r in ruedas_unicas)
        / len(ruedas_unicas)
        if ruedas_unicas else 0.0
    )

    return {
        "ejes": ejes,
        "ruedas": ruedas_unicas,
        "confianza_ejes": round(confianza, 4)
    }


# =====================================
# INFERENCIA UNIFICADA
# =====================================

def procesar_imagen(imagen):

    inicio = time.perf_counter()

    t1 = time.perf_counter()

    m1 = inferencia_m1(imagen)

    latencia_m1 = (
        time.perf_counter() - t1
    ) * 1000

    t2 = time.perf_counter()

    m2 = inferencia_m2(imagen)

    latencia_m2 = (
        time.perf_counter() - t2
    ) * 1000

    t3 = time.perf_counter()

    m3 = inferencia_m3(imagen, m2)

    latencia_m3 = (
        time.perf_counter() - t3
    ) * 1000

    latencia_total = (
        time.perf_counter() - inicio
    ) * 1000

    return {
        "placa": m1["placa"],
        "confianza": m1["confianza"],
        "confianza_ocr": m1["confianza"],
        "formato_valido": m1["formato_valido"],
        "categoria": m2["categoria"],
        "confianza_categoria": m2["confianza_categoria"],
        "ejes": m3["ejes"],
        "confianza_ejes": m3["confianza_ejes"],
        "latencia_ms": round(latencia_total, 2),

        "latencias": {
            "m1_ms": round(latencia_m1, 2),
            "m2_ms": round(latencia_m2, 2),
            "m3_ms": round(latencia_m3, 2)
        },

        "detecciones": {
            "placa": m1,
            "vehiculo": m2,
            "ruedas": m3["ruedas"]
        }
    }


print("FUNCIONES DE INFERENCIA LISTAS")

FUNCIONES DE INFERENCIA LISTAS


In [10]:
import secrets
import threading
import time
import requests
import uvicorn
from fastapi import FastAPI, File, UploadFile, Header, HTTPException

# Detener una instancia anterior si existe.
if "servidor" in globals():
    servidor.should_exit = True
    if "hilo_servidor" in globals():
        hilo_servidor.join(timeout=15)
        assert not hilo_servidor.is_alive(), "Reinicia la sesión: servidor anterior ocupado"

API_TOKEN = secrets.token_urlsafe(32)
INFERENCE_LOCK = threading.Lock()
app = FastAPI(title="API Inteligente de Peaje")

def autorizar(authorization):
    if not secrets.compare_digest(authorization or "", "Bearer " + API_TOKEN):
        raise HTTPException(status_code=401, detail="Token inválido")

def leer_imagen(file):
    contenido = file.file.read(10 * 1024 * 1024 + 1)
    if len(contenido) > 10 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Imagen demasiado grande")
    imagen = cv2.imdecode(np.frombuffer(contenido, np.uint8), cv2.IMREAD_COLOR)
    if imagen is None:
        raise HTTPException(status_code=400, detail="Imagen inválida")
    return imagen

@app.get("/")
def inicio():
    return {"servicio": "API Inteligente de Peaje"}

@app.get("/health")
def health():
    return {"estado": "online", "m1": "cargado", "m2": "cargado", "m3": "cargado", "gpu": torch.cuda.is_available()}

@app.post("/predict")
def predict(file: UploadFile = File(...), authorization: str = Header(default="")):
    autorizar(authorization)
    imagen = leer_imagen(file)
    with INFERENCE_LOCK:
        return procesar_imagen(imagen)

@app.post("/ocr")
def ocr(file: UploadFile = File(...), authorization: str = Header(default="")):
    autorizar(authorization)
    imagen = leer_imagen(file)
    with INFERENCE_LOCK:
        inicio = time.perf_counter()
        resultado = inferencia_m1(imagen)
        resultado["latencia_ms"] = round((time.perf_counter() - inicio) * 1000, 2)
        return resultado

servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
hilo_servidor.start()
for intento in range(30):
    if servidor.started:
        break
    if not hilo_servidor.is_alive():
        raise RuntimeError("FastAPI no pudo iniciar; revisa el log anterior")
    time.sleep(1)
assert servidor.started, "FastAPI no inició a tiempo"
respuesta = requests.get("http://127.0.0.1:8000/health", timeout=10)
respuesta.raise_for_status()
print("FASTAPI ACTIVO:", respuesta.json())


INFO:     Started server process [4327]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52106 - "GET /health HTTP/1.1" 200 OK
FASTAPI ACTIVO: {'estado': 'online', 'm1': 'cargado', 'm2': 'cargado', 'm3': 'cargado', 'gpu': False}


In [11]:
!pip install -q pyngrok

from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
from google.colab import userdata
import requests
import time

# Verificar FastAPI

respuesta = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

assert respuesta.status_code == 200

print("FastAPI funcionando:", respuesta.json())

# Recuperar token de Colab

NGROK_TOKEN = userdata.get("NGROK_TOKEN")

try:
    ngrok.set_auth_token(NGROK_TOKEN)

except PyngrokNgrokError as e:

    if "Authtoken already set" in str(e):
        print("Token configurado correctamente")
    else:
        raise

# Cerrar túneles anteriores

ngrok.kill()

# Crear túnel público

tunel = ngrok.connect(
    addr=8000,
    proto="http"
)

INFERENCE_URL = tunel.public_url

time.sleep(3)

respuesta = requests.get(
    INFERENCE_URL + "/health",
    headers={
        "ngrok-skip-browser-warning": "true"
    },
    timeout=30
)

assert respuesta.status_code == 200

print("\n" + "=" * 60)
print("API PUBLICADA CORRECTAMENTE")
print("INFERENCE_URL =", INFERENCE_URL)
print("SWAGGER =", INFERENCE_URL + "/docs")
print("HEALTH =", INFERENCE_URL + "/health")
print("PREDICT =", INFERENCE_URL + "/predict")
print("OCR =", INFERENCE_URL + "/ocr")
print("=" * 60)

print(respuesta.json())

INFO:     127.0.0.1:45314 - "GET /health HTTP/1.1" 200 OK
FastAPI funcionando: {'estado': 'online', 'm1': 'cargado', 'm2': 'cargado', 'm3': 'cargado', 'gpu': False}
INFO:     34.135.61.0:0 - "GET /health HTTP/1.1" 200 OK

API PUBLICADA CORRECTAMENTE
INFERENCE_URL = https://conjecturably-metabiotic-santana.ngrok-free.dev
SWAGGER = https://conjecturably-metabiotic-santana.ngrok-free.dev/docs
HEALTH = https://conjecturably-metabiotic-santana.ngrok-free.dev/health
PREDICT = https://conjecturably-metabiotic-santana.ngrok-free.dev/predict
OCR = https://conjecturably-metabiotic-santana.ngrok-free.dev/ocr
{'estado': 'online', 'm1': 'cargado', 'm2': 'cargado', 'm3': 'cargado', 'gpu': False}


In [12]:
print("INFERENCE_URL =", INFERENCE_URL)
print("INFERENCE_TOKEN =", API_TOKEN)


INFERENCE_URL = https://conjecturably-metabiotic-santana.ngrok-free.dev
INFERENCE_TOKEN = DzH-Rr_OnGx4TLn9L8sTiR-PWZ_DkFoVg3g5qu1coZ4


In [ ]:
print("INFERENCE_URL =", INFERENCE_URL)
print("INFERENCE_TOKEN =", API_TOKEN)

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import json

subidos = files.upload()

nombre = next(iter(subidos))

imagen = cv2.imdecode(
    np.frombuffer(subidos[nombre], np.uint8),
    cv2.IMREAD_COLOR
)

assert imagen is not None, "Imagen inválida"

resultado = procesar_imagen(imagen)

print(json.dumps(
    resultado,
    indent=4,
    ensure_ascii=False
))

anotada = imagen.copy()

vehiculo = resultado["detecciones"]["vehiculo"]

placa = resultado["detecciones"]["placa"]

ruedas = resultado["detecciones"]["ruedas"]

# Dibujar segmentación M2

if vehiculo["mascara"]:

    poligono = np.array(
        vehiculo["mascara"],
        dtype=np.int32
    )

    capa = anotada.copy()

    cv2.fillPoly(
        capa,
        [poligono],
        (255, 100, 0)
    )

    anotada = cv2.addWeighted(
        capa, 0.25,
        anotada, 0.75,
        0
    )

# Dibujar vehículo

if vehiculo["bbox"]:

    x1, y1, x2, y2 = vehiculo["bbox"]

    cv2.rectangle(
        anotada,
        (x1, y1),
        (x2, y2),
        (255, 100, 0),
        3
    )

    cv2.putText(
        anotada,
        str(resultado["categoria"]),
        (x1, max(25, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 100, 0),
        2
    )

# Dibujar placa

if placa["bbox"]:

    x1, y1, x2, y2 = placa["bbox"]

    cv2.rectangle(
        anotada,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        3
    )

    cv2.putText(
        anotada,
        str(resultado["placa"]),
        (x1, max(25, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

# Dibujar centros de ruedas

for i, rueda in enumerate(ruedas, 1):

    x = int(rueda["x"])
    y = int(rueda["y"])

    cv2.circle(
        anotada,
        (x, y),
        8,
        (0, 0, 255),
        -1
    )

    cv2.putText(
        anotada,
        f"Eje {i}",
        (x + 10, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 255),
        2
    )

plt.figure(figsize=(14, 9))

plt.imshow(
    cv2.cvtColor(
        anotada,
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    f"Placa: {resultado['placa']} | "
    f"Categoria: {resultado['categoria']} | "
    f"Ejes: {resultado['ejes']}"
)

plt.axis("off")

plt.show()

cv2.imwrite(
    str(BASE / "evidencias/prueba_inferencia.jpg"),
    anotada
)

with open(
    BASE / "evidencias/prueba_inferencia.json",
    "w"
) as f:

    json.dump(
        resultado,
        f,
        indent=4,
        ensure_ascii=False
    )

print("EVIDENCIAS GUARDADAS")

In [ ]:
import requests
import json

with open(nombre, "rb") as archivo:

    respuesta = requests.post(
        INFERENCE_URL + "/predict",
        files={"file": archivo},
        headers={
            "Authorization": f"Bearer {API_TOKEN}",
            "ngrok-skip-browser-warning": "true"
        },
        timeout=120
    )

print("Código HTTP:", respuesta.status_code)

print(json.dumps(
    respuesta.json(),
    indent=4,
    ensure_ascii=False
))

assert respuesta.status_code == 200

print("API DE INFERENCIA FUNCIONANDO")